# Figure 04 - clustering and representative-link validation

This notebook follows the validation workflow described in the thesis and implemented in `python/tools/viasat_analysis.py`:

1. compute link penetration from Viasat hourly sums and mean simulated daily flows;
2. use KMeans on the log penetration coefficient for the first clustering step;
3. show the three archived dendrogram subclusters of the representative major cluster;
4. select one low-error, spatially distributed representative polyline per subcluster and validate it.

No network-wide hourly sum is plotted: the link-level comparisons are the meaningful validation quantity here.

In [ ]:
# Paper palette inherited from regenerate_figures_03_validation.ipynb, with a compact Nature-like style.
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm

PALETTE = {
    "src_dest": "#E7D1D1",
    "rnd_rnd": "#DCC7DF",
    "rnd_dest": "#C4DADD",
    "src_rnd": "#C8DEC8",
    "roads_main": "#8D8D8D",
    "roads_minor": "#CFCFCF",
    "text": "#2F2F2F",
    "background": "#FAFAFA",
    "grid": "#DDDDDD",
}

paper_cmap = LinearSegmentedColormap.from_list(
    "paper_activity",
    ["#EAF6F7", "#AFCFD4", "#D5C0DC", "#D9B8B8", "#9E5F63"],
    N=256,
)
SUBCLUSTER_COLORS = {
    0: "#343434",  # charcoal, distinct from the penetration-cluster black
    1: "#C4673E",  # terracotta for the historically red subcluster
    2: "#25818E",  # muted teal for the historically blue subcluster
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.titlesize": 10, "axes.titleweight": "bold", "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "figure.titlesize": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#4D4D4D", "axes.linewidth": 0.8, "axes.grid": False,
    "grid.color": "#D9D9D9", "grid.alpha": 0.65, "grid.linewidth": 0.5,
    "legend.frameon": False,
    "lines.linewidth": 1.5, "xtick.major.size": 3.5, "ytick.major.size": 3.5,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "figure.dpi": 300, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
})

In [ ]:
from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.cluster.hierarchy as sch
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

try:
    import contextily as cx
except ImportError:
    cx = None

ROOT = Path.cwd()
if not (ROOT / "article").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "article").exists())

IMAGES_DIR = ROOT / "article/tex/images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
YEAR = 2020
HOUR_LABELS = [f"{hour:02d}:00" for hour in range(24)]
HOUR_TICKS = np.arange(0, 24, 3)

VIASAT_PATH = ROOT / "work_viasat/viasat_20200210_20200218_timed_fluxes.csv"
UNIQ_DIR = ROOT / "work_viasat/sim2020/uniq"
INNER_CLUSTER_PATH = ROOT / "work_viasat/viasat_data_analysis/2020/None/inner_cluster/inner_cluster_2.csv"

# Historical thesis examples identify the KMeans major cluster.
THESIS_REPRESENTATIVE_IDS = [104671, 18596, 190978]
# Keep the urban black representative for its interpretable location near Forlimpopoli.
FIXED_BLACK_REPRESENTATIVE_CID = 104671
REPRESENTATIVE_STEMS = {0: "8a_cluster_black", 1: "8c_cluster_red", 2: "8b_cluster_blue"}
MAX_REPRESENTATIVE_L2 = 0.80
MIN_REPRESENTATIVE_SEPARATION_KM = 40.0

def save_figure(fig, stem):
    for extension in ("png", "pdf"):
        fig.savefig(IMAGES_DIR / f"{stem}.{extension}", bbox_inches="tight")
    print(f"saved: article/tex/images/{stem}.png")

## Source data and thesis preprocessing

The historical implementation sums observed Viasat flows over the validation interval, averages corrected daily simulation outputs, and shifts simulated timestamps by 75 minutes before hourly aggregation. The shift is preserved because omitting it changes the three reported L2 results.

In [ ]:
def load_viasat_hourly(path):
    raw = pd.read_csv(path, sep=";", parse_dates=["time"])
    raw = raw.set_index("time").sort_index()
    start_date, stop_date = raw.index[0], raw.index[-1]
    raw = raw[(raw.index >= start_date) & (raw.index < stop_date)].copy()
    raw["time_hour"] = raw.index.time
    hourly = raw.groupby(["time_hour", "cid"])["total_fluxes"].sum().unstack(fill_value=0)
    return hourly, start_date, stop_date

def load_simulation_hourly(directory):
    frames = []
    paths = sorted(directory.glob("*.csv.corrected"))
    if not paths:
        raise FileNotFoundError(f"No corrected simulation files in {directory}")
    for path in paths:
        daily = pd.read_csv(path, sep=";").groupby("poly_cid").sum(numeric_only=True)
        daily = daily.drop(columns=["poly_lid"], errors="ignore")
        parsed = pd.to_datetime(daily.columns, errors="coerce")
        daily = daily.loc[:, ~parsed.isna()]
        daily.columns = parsed[~parsed.isna()]
        hourly = daily.T
        hourly.index = hourly.index + pd.Timedelta(minutes=75)
        hourly["hour"] = hourly.index.hour
        frames.append(hourly.groupby("hour").sum().reindex(range(24), fill_value=0))
    return pd.concat(frames, keys=range(len(frames))).groupby(level=1).mean()

viasat_df, start_date, stop_date = load_viasat_hourly(VIASAT_PATH)
simulation_df = load_simulation_hourly(UNIQ_DIR)
simulation_df.index = viasat_df.index

common_cids = sorted(set(viasat_df.columns).intersection(simulation_df.columns))
viasat_df = viasat_df[common_cids]
simulation_df = simulation_df[common_cids]
print(f"Validation interval: {start_date.date()} to {stop_date.date()}")
print(f"Common links: {len(common_cids)}; corrected simulations: {len(list(UNIQ_DIR.glob('*.csv.corrected')))}")

## First clustering step: statistical penetration

As in `viasat_analysis.py`, KMeans is applied to the rounded logarithm of the Viasat/simulation penetration coefficient. KMeans label integers are arbitrary; the representative major cluster is identified as the label containing the three thesis representative polylines.

In [ ]:
ratio = viasat_df.sum(axis=0) / simulation_df.sum(axis=0)
ratio = ratio[(ratio > 0) & np.isfinite(ratio)]
penetration = pd.DataFrame({"log_penetration": np.log10(ratio)})
penetration["rounded_log_penetration"] = penetration["log_penetration"].round().astype(int)

features = StandardScaler().fit_transform(penetration)
kmeans = KMeans(init="random", n_clusters=3, n_init=10, random_state=1)
penetration["major_cluster"] = kmeans.fit_predict(features)

representative_major_labels = penetration.loc[THESIS_REPRESENTATIVE_IDS, "major_cluster"]
major_cluster = representative_major_labels.mode().iat[0]
major_cids = penetration.index[penetration["major_cluster"] == major_cluster].tolist()
print("Major-cluster labels for the representative roads:")
print(representative_major_labels.to_string())
print(f"Representative major cluster label: {major_cluster}; links: {len(major_cids)}")

## Dendrogram subclusters and spatial location

The three subcluster assignments are read from the historical dendrogram output saved for the thesis. This keeps the representative-road interpretation stable while recreating the figure with the current paper palette.

In [ ]:
from scipy.cluster.hierarchy import fcluster

# ---------------------------------------------------------------------
# Load inner clusters
# ---------------------------------------------------------------------
inner_clusters = pd.read_csv(INNER_CLUSTER_PATH, sep=";")
inner_clusters["geometry"] = gpd.GeoSeries.from_wkt(inner_clusters["geometry"])
inner_gdf = gpd.GeoDataFrame(
    inner_clusters,
    geometry="geometry",
    crs="EPSG:4326",
)

# ---------------------------------------------------------------------
# Select three low-error representatives with broad spatial coverage
# ---------------------------------------------------------------------
selection_gdf = inner_gdf.to_crs(epsg=3857).copy()
selection_gdf["midpoint"] = selection_gdf.geometry.interpolate(0.5, normalized=True)
selection_gdf["x"] = selection_gdf["midpoint"].x
selection_gdf["y"] = selection_gdf["midpoint"].y

candidate_rows = []
for row in selection_gdf.itertuples():
    cid = int(row.cid)
    if cid not in simulation_df.columns or cid not in viasat_df.columns:
        continue
    simulation = simulation_df[cid].astype(float)
    observed = viasat_df[cid].astype(float)
    if simulation.sum() <= 0 or observed.sum() <= 0:
        continue
    incremented = observed / (observed.sum() / simulation.sum())
    hourly_l2 = 100 * np.abs(
        simulation / simulation.sum() - incremented / incremented.sum()
    )
    candidate_rows.append({
        "cid": cid,
        "cluster": int(row.cluster),
        "mean_L2_percent": float(hourly_l2.mean()),
        "sim_total": float(simulation.sum()),
        "observed_total": float(observed.sum()),
        "x": float(row.x),
        "y": float(row.y),
    })

candidate_metrics = pd.DataFrame(candidate_rows)
eligible = candidate_metrics.query(
    "mean_L2_percent <= @MAX_REPRESENTATIVE_L2 and sim_total >= 300 and observed_total >= 50"
).copy()
fixed_black = eligible.loc[eligible["cid"] == FIXED_BLACK_REPRESENTATIVE_CID]
if len(fixed_black) != 1 or int(fixed_black.iloc[0]["cluster"]) != 0:
    raise ValueError("The fixed urban black representative is not eligible in subcluster 1")
candidate_pools = [
    fixed_black.to_dict("records"),
    eligible.loc[eligible["cluster"] == 1].to_dict("records"),
    eligible.loc[eligible["cluster"] == 2].to_dict("records"),
]

solutions = []
for selected in itertools.product(*candidate_pools):
    distances = [
        np.hypot(a["x"] - b["x"], a["y"] - b["y"]) / 1000
        for a, b in itertools.combinations(selected, 2)
    ]
    if min(distances) >= MIN_REPRESENTATIVE_SEPARATION_KM:
        solutions.append({
            "selected": selected,
            "total_L2": sum(item["mean_L2_percent"] for item in selected),
            "min_distance_km": min(distances),
            "distance_sum_km": sum(distances),
        })

if not solutions:
    raise ValueError("No spatially distributed representative-polyline solution satisfies the constraints")

best_solution = min(solutions, key=lambda item: (item["total_L2"], -item["distance_sum_km"]))
REPRESENTATIVES = {
    int(item["cluster"]): {
        "cid": int(item["cid"]),
        "label": f"Subcluster {int(item['cluster']) + 1}",
        "stem": REPRESENTATIVE_STEMS[int(item["cluster"])],
    }
    for item in best_solution["selected"]
}
REPRESENTATIVE_IDS = [REPRESENTATIVES[cluster]["cid"] for cluster in range(3)]
selected_metrics = candidate_metrics.loc[candidate_metrics["cid"].isin(REPRESENTATIVE_IDS)].sort_values("cluster")
print("Selected representative links (fixed urban black link; low L2 and spatial separation for the others):")
print(selected_metrics[["cluster", "cid", "mean_L2_percent"]].to_string(index=False))
print(f"Minimum pairwise distance: {best_solution['min_distance_km']:.1f} km")

print(
    inner_clusters
    .groupby("cluster")
    .size()
    .rename("number_of_links")
    .to_string()
)

print(
    inner_clusters[
        inner_clusters["cid"].isin(REPRESENTATIVE_IDS)
    ][["cid", "cluster"]]
    .sort_values("cluster")
    .to_string(index=False)
)

# ---------------------------------------------------------------------
# Prepare curves for dendrogram
# ---------------------------------------------------------------------
historic_major_cids = [
    cid for cid in inner_clusters["cid"]
    if cid in simulation_df.columns
]

curves = simulation_df[historic_major_cids].T.values

linkage = sch.linkage(
    curves,
    method="ward",
    metric="euclidean",
)

# Map each cid to its geolocated subcluster
cid_to_map_cluster = (
    inner_clusters
    .set_index("cid")
    .loc[historic_major_cids, "cluster"]
    .astype(int)
    .to_dict()
)

# ---------------------------------------------------------------------
# Dendrogram coloring consistent with geographical clusters
# ---------------------------------------------------------------------
cut_height = 15000
n_obs = len(historic_major_cids)

node_leaves = {i: [i] for i in range(n_obs)}

for i, row in enumerate(linkage):
    node_id = n_obs + i
    left = int(row[0])
    right = int(row[1])
    node_leaves[node_id] = node_leaves[left] + node_leaves[right]


def link_color_func(node_id):
    leaves = node_leaves[node_id]

    map_clusters = {
        cid_to_map_cluster[historic_major_cids[i]]
        for i in leaves
    }

    if len(map_clusters) == 1:
        cluster = list(map_clusters)[0]
        return SUBCLUSTER_COLORS[int(cluster)]

    return "#545454"


# ---------------------------------------------------------------------
# Plot dendrogram
# ---------------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(6.8, 3.7),
    constrained_layout=True,
)

sch.dendrogram(
    linkage,
    no_labels=True,
    color_threshold=cut_height,
    above_threshold_color="#545454",
    link_color_func=link_color_func,
    ax=ax,
)

ax.axhline(
    cut_height,
    color="#545454",
    linestyle="--",
    linewidth=1.2,
    label="Cut height = 15,000",
)

ax.set_title(
    "Hierarchical clustering of representative links",
    loc="left",
)

ax.set_xlabel("Polylines")
ax.set_ylabel("Ward distance")
ax.grid(axis="y", zorder=0)
ax.legend(loc="upper right")

save_figure(fig, "7a_dendrogram")
plt.show()

# ---------------------------------------------------------------------
# Geographical map of subclusters
# ---------------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(6.4, 5.2),
    constrained_layout=True,
)

map_gdf = inner_gdf.to_crs(epsg=3857)

for cluster, group in map_gdf.groupby("cluster"):
    cluster = int(cluster)

    group.plot(
        ax=ax,
        color=SUBCLUSTER_COLORS[cluster],
        linewidth=2.1,
        alpha=0.98,
        zorder=4,
        label=f"Subcluster {cluster + 1}",
    )

if cx is not None:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            cx.add_basemap(
                ax,
                source=cx.providers.CartoDB.Positron,
                attribution=False,
                alpha=0.55,
                zorder=0,
            )
        except Exception as exc:
            print(f"Basemap unavailable: {exc}")

representative_map = map_gdf[
    map_gdf["cid"].isin(REPRESENTATIVE_IDS)
]

representative_map.plot(
    ax=ax,
    color="white",
    linewidth=4.6,
    zorder=5,
)

for cluster, group in representative_map.groupby("cluster"):
    cluster = int(cluster)

    group.plot(
        ax=ax,
        color=SUBCLUSTER_COLORS[cluster],
        linewidth=3.2,
        zorder=6,
    )

ax.set_title(
    "Geographical distribution of subclusters",
    loc="left",
)

ax.set_axis_off()
ax.legend(loc="lower left", handlelength=2.4)

save_figure(fig, "7b_dendrogram_geoloc")
plt.show()

## Representative-polyline validation

Observed Viasat flows are incremented separately for each polyline using its own penetration coefficient. This avoids the invalid interpretation of thousands of vehicles per hour on an aggregated set of roads.

In [ ]:
def plot_representative_validation(subcluster, entry):
    cid = entry["cid"]
    simulation = simulation_df[cid].astype(float)
    observed = viasat_df[cid].astype(float)
    penetration_coefficient = observed.sum() / simulation.sum()
    incremented = observed / penetration_coefficient
    l2_hourly = 100 * np.abs(
        simulation / simulation.sum() - incremented / incremented.sum()
    )

    x = np.arange(24)
    cluster_color = SUBCLUSTER_COLORS[subcluster]
    fig, (ax_top, ax_bottom) = plt.subplots(
        2, 1, figsize=(6.5, 4.7), sharex=True,
        gridspec_kw={"height_ratios": [1.75, 1.0], "hspace": 0.08}, layout="constrained",
    )
    ax_top.plot(x, simulation.values, color=cluster_color, label="Simulation")
    ax_top.plot(x, incremented.values, color="#6E6E6E", linestyle=(0, (4, 2)), label="Viasat incremented")
    ax_top.set_ylabel(r"$\Phi$", rotation=0)
    ax_top.legend(loc="upper left", ncol=2, handlelength=2.4)
    ax_top.set_title(f"{entry['label']} | Polyline {cid}", loc="left")

    bars = ax_bottom.bar(
        x,
        l2_hourly.values,
        color=cluster_color,
        alpha=0.88,
        width=0.62,
        edgecolor="white",
        linewidth=0.7,
        zorder=3,
    )

    mean_l2 = l2_hourly.mean()

    ax_bottom.axhline(
        mean_l2,
        color="#333333",
        linestyle="--",
        linewidth=1.2,
        alpha=0.85,
        zorder=4,
    )

    ax_bottom.fill_between(
        x,
        0,
        l2_hourly.values,
        color=cluster_color,
        alpha=0.08,
        zorder=1,
    )

    ax_bottom.set_ylabel("L2 error (%)")
    ax_bottom.set_xlabel("Time of day")
    ax_bottom.set_xticks(HOUR_TICKS)
    ax_bottom.set_xticklabels(
        [HOUR_LABELS[i] for i in HOUR_TICKS],
        rotation=45,
        ha="right",
    )

    ax_bottom.set_ylim(0, max(4.0, l2_hourly.max() * 1.18))

    ax_bottom.grid(
        True,
        axis="y",
        color="#D0D0D0",
        linewidth=0.7,
        alpha=0.65,
        zorder=0,
    )

    ax_bottom.spines["top"].set_visible(False)
    ax_bottom.spines["right"].set_visible(False)

    ax_bottom.text(
        0.98,
        0.90,
        f"Mean L2 = {mean_l2:.2f}%",
        transform=ax_bottom.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        color="#333333",
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="#DDDDDD",
            alpha=0.9,
        ),
    )

    save_figure(fig, entry["stem"])
    plt.show()
    return {
        "subcluster": subcluster,
        "polyline": cid,
        "penetration": penetration_coefficient,
        "mean_L2_percent": l2_hourly.mean(),
    }

validation_results = pd.DataFrame([
    plot_representative_validation(subcluster, entry)
    for subcluster, entry in REPRESENTATIVES.items()
])
validation_results

In [ ]:
# 8d_representative_polylines_location.png - detailed location of the three validated links.
try:
    representative_map = inner_gdf.loc[
        inner_gdf["cid"].isin(REPRESENTATIVE_IDS)
    ].to_crs(epsg=3857).copy()

    if len(representative_map) != len(REPRESENTATIVE_IDS):
        raise ValueError("All three representative polylines must have a geometry record")

    representative_map["color"] = representative_map["cluster"].map(SUBCLUSTER_COLORS)
    xmin, ymin, xmax, ymax = representative_map.total_bounds
    buffer = 7000

    local_roads = map_gdf.cx[xmin - buffer:xmax + buffer, ymin - buffer:ymax + buffer]
    label_offsets = {0: (8, 10), 1: (8, -15), 2: (8, 10)}

    fig, ax = plt.subplots(figsize=(7.2, 5.6), dpi=300, constrained_layout=True)
    ax.set_xlim(xmin - buffer, xmax + buffer)
    ax.set_ylim(ymin - buffer, ymax + buffer)

    if cx is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            cx.add_basemap(
                ax,
                source=cx.providers.CartoDB.Positron,
                crs=representative_map.crs,
                attribution=False,
                zoom=8,
                alpha=0.60,
                zorder=0,
            )

    local_roads.plot(ax=ax, color="#CBCBCB", linewidth=0.8, alpha=0.55, zorder=2)

    for cluster, group in representative_map.groupby("cluster"):
        cluster = int(cluster)
        group.plot(ax=ax, color="white", linewidth=6.2, zorder=4)
        group.plot(
            ax=ax,
            color=SUBCLUSTER_COLORS[cluster],
            linewidth=3.8,
            zorder=5,
            label=f"Subcluster {cluster + 1}",
        )

    for _, row in representative_map.iterrows():
        cid = int(row["cid"])
        midpoint = row.geometry.interpolate(0.5, normalized=True)
        ax.annotate(
            f"Polyline {cid}",
            xy=(midpoint.x, midpoint.y),
            xytext=label_offsets[int(row["cluster"])],
            textcoords="offset points",
            fontsize=8,
            fontweight="bold",
            color=SUBCLUSTER_COLORS[int(row["cluster"])],
            bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#DDDDDD", "alpha": 0.94},
            zorder=6,
        )

    ax.set_title("Representative validation links", loc="left")
    ax.legend(loc="lower left", handlelength=2.4)
    ax.set_axis_off()
    save_figure(fig, "8d_representative_polylines_location")
    plt.show()
    print("OK: geolocated representative polylines:", sorted(REPRESENTATIVE_IDS))
except Exception as exc:
    print(f"ERROR representative_polylines_location: {type(exc).__name__}: {exc}")
    raise